<span style="color:lightgreen">

# L4.3 Transcripción con Traducción del portugués al inglés en cascada Covost2 pt-en

<span>

# Cascade ST Baseline experiment using Whisper along with NLLB and Europarl-ST (Spanish to English)

In this notebook, we are going to learn how to use Whisper and Meta's large pre-trained models [NLLB](https://huggingface.co/docs/transformers/model_doc/nllb) in a cascade ST approach on the Europarl-ST using the English-Spanish translation pair.

First, we import automatic transcriptions, references and translations (and their clean counterparts) generated by the ASR baseline experiment using Whisper

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
from datasets import load_dataset

# NOTE: L4,1 tine datos de transcripción de test
# Cuando lo cargamos la partición se llama train pero no es train es test
raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv"))

# Filter out rows with null values in hypothesis or translation
raw_datasets = raw_datasets.filter(lambda x: x["hypothesis"] is not None and x["translation"] is not None)

print(raw_datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/4023 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'hypothesis', 'reference', 'translation', 'hypothesis_clean', 'reference_clean', 'translation_clean'],
        num_rows: 4023
    })
})


Show the first sample for each dataset feature

In [3]:
print(raw_datasets["train"][:1]["hypothesis"])
print(raw_datasets["train"][:1]["reference"])
print(raw_datasets["train"][:1]["translation"])
print(raw_datasets["train"][:1]["hypothesis_clean"])
print(raw_datasets["train"][:1]["reference_clean"])
print(raw_datasets["train"][:1]["translation_clean"])

[' e dia de ir emprestado as pessoas daudei.']
['Pedir dinheiro emprestado às pessoas da aldeia']
['Borrow money from people in the village']
[' e dia de ir emprestado as pessoas daudei ']
['pedir dinheiro emprestado às pessoas da aldeia']
['borrow money from people in the village']


<p style="page-break-after:always;"></p>

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Spanish-English pair:

In [4]:
max_tok_length = 275

from transformers import AutoTokenizer

checkpoint = "facebook/nllb-200-distilled-600M"
# from flores200_codes import flores_codes
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    src_lang=CONFIG.src_code, 
    tgt_lang=CONFIG.tgt_code, 
    truncation=True, 
    max_length=max_tok_length,
    )

We will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be used. In this case we tokenize speech transcriptions and translations to perform ST cascade experiments

In [5]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["hypothesis"], 
        text_target = sample["translation"],
        truncation=True,
        max_length=max_tok_length
        )
    return model_inputs

Now, we can apply the preprocess_function to the raw datasets

In [6]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/4023 [00:00<?, ? examples/s]

<p style="page-break-after:always;"></p>

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>

In [7]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [8]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )

<p style="page-break-after:always;"></p>

## Inference

At inference time, it is recommended to use the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation). This method takes care of encoding the input and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers.
There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB.

In [9]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



We prepare the dataset in batches to be translated:

In [10]:
test_batch_size = 32
# NOTE: L4,1 tine datos de transcripción de test
# Cuando lo cargamos la partición se llama train pero no es train es test
batch_tokenized_test = tokenized_datasets['train'].batch(test_batch_size)

Batching examples:   0%|          | 0/4023 [00:00<?, ? examples/s]

<p style="page-break-after:always;"></p>

Processing in batches to add padding and convert to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

In [11]:
from tqdm import tqdm

number_of_batches = len(batch_tokenized_test["hypothesis"])
output_sequences = []
for i in tqdm(range(number_of_batches)):
    inputs = tokenizer(
        batch_tokenized_test["hypothesis"][i], 
        max_length=max_tok_length, 
        truncation=False, 
        return_tensors="pt", 
        padding=True,
        )
    output_batch = model.generate(
        generation_config=generation_config, 
        input_ids=inputs["input_ids"].cuda(), 
        attention_mask=inputs["attention_mask"].cuda(), 
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), 
        max_length = max_tok_length, 
        num_beams=1, 
        do_sample=False,
        )
    output_sequences.extend(output_batch.cpu())

  0%|                                                                                                                                                     | 0/126 [00:00<?, ?it/s]

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2914: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


  1%|█                                                                                                                                            | 1/126 [00:00<00:48,  2.60it/s]

  2%|██▏                                                                                                                                          | 2/126 [00:00<00:32,  3.86it/s]

  2%|███▎                                                                                                                                         | 3/126 [00:00<00:28,  4.39it/s]

...

 98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋   | 123/126 [00:32<00:00,  4.50it/s]

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊  | 124/126 [00:32<00:00,  4.77it/s]

 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉ | 125/126 [00:33<00:00,  4.85it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126/126 [00:33<00:00,  4.73it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 126/126 [00:33<00:00,  3.78it/s]

We can recover the decoded predictions and references by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [12]:
decoded_preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

In [13]:
# NOTE: L4,1 tine datos de transcripción de test
# Cuando lo cargamos la partición se llama train pero no es train es test
references = tokenizer.batch_decode(tokenized_datasets["train"]["labels"], skip_special_tokens=True)

Let's take a closer look at some decoded predictions and references:

In [14]:
decoded_preds[:1]

['And the day you borrow, people will be thrilled.']

In [15]:
references[:2]

['Borrow money from people in the village', 'Lock them up']

<p style="page-break-after:always;"></p>

Predictions and references are normalized using the Whisper basic text standardisation/normalization module

In [16]:
from whisper.normalizers.basic import BasicTextNormalizer

normalizer = BasicTextNormalizer()

decoded_preds_clean = [normalizer(text) for text in decoded_preds]
references_clean = [normalizer(text) for text in references]

For evaluation, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [17]:
from evaluate import load

metric = load("sacrebleu")

In [18]:
result = metric.compute(predictions=decoded_preds_clean, references=references_clean)
print(f'BLEU score: {result["score"]:.1f}')

BLEU score: 34.2


Compute COMET figures using the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics.

In [19]:
from evaluate import load
comet_metric = load('comet')

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [20]:
comet_score = comet_metric.compute(predictions=decoded_preds_clean, references=references_clean, sources=raw_datasets["train"]["hypothesis"])

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
print(f"COMET: {comet_score['mean_score'] * 100:.2f} %")

COMET: 75.72 %
